# Anomaly Detection — ERFNet (MSP (different t value)· MaxLogit · MaxEntropy)


## Cell 1 — Mount Drive & verify GPU

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/AnomalyProject/results', exist_ok=True)

gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print('Drive mounted.')
print('GPU:', gpu if gpu else '❌ NOT FOUND — change runtime to T4 GPU!')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
GPU: Tesla T4


## Cell 2 — Install packages & clone repo

In [ ]:
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'gdown', 'Pillow', 'numpy',
                'torch', 'torchvision'], check=True)
print('✓ Packages installed')

REPO = '/content/MaskArchitectureAnomaly_CourseProject'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone',
        'https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject.git'],
        check=True)
print('✓ Repo ready')
print('eval/ contents:', os.listdir(os.path.join(REPO, 'eval')))

✓ Packages installed
✓ Repo ready
eval/ contents: ['eval_iou.py', 'results.txt', 'iouEval.py', 'README.md', 'dataset.py', '__pycache__', 'erfnet_nobn.py', 'evalAnomaly.py', 'erfnet.py', 'eval_forwardTime.py', 'eval_cityscapes_server.py', 'eval_cityscapes_color.py', 'metrics.py', 'transform.py']


## Cell 3 — Download & extract datasets
Downloads `Anomaly_Validation_Datasets.zip` from the course Drive folder.

In [ ]:
import subprocess, sys, os, zipfile

DATASET_DIR = '/content/datasets'
ZIP_PATH    = '/content/Anomaly_Validation_Datasets.zip'
FOLDER_ID   = '1q2vHUzora2nP52fP50zmoQAykWuwoGav'

os.makedirs(DATASET_DIR, exist_ok=True)

# ── Step 1: list folder to find zip file ID ────────────────────────────────
if not os.path.isfile(ZIP_PATH):
    print('Listing Drive folder to find zip ID...')
    r = subprocess.run(
        ['gdown', '--folder', FOLDER_ID, '--list', '--remaining-ok'],
        capture_output=True, text=True)
    print(r.stdout)

    # Parse the zip file ID from gdown output
    zip_id = None
    for line in r.stdout.splitlines():
        if 'Anomaly_Validation_Datasets' in line and 'id:' in line:
            zip_id = line.split('id:')[-1].strip()
            break

    if zip_id is None:
        # Fallback: try downloading entire folder (zip may be at top level)
        print('Could not auto-detect zip ID — downloading full folder...')
        subprocess.run(['gdown', '--folder', FOLDER_ID,
                        '-O', '/content/drive_dl', '--remaining-ok'], check=True)
        for root, _, files in os.walk('/content/drive_dl'):
            for f in files:
                if f.endswith('.zip'):
                    import shutil
                    shutil.move(os.path.join(root, f), ZIP_PATH)
                    print(f'Found zip: {f}')
                    break
    else:
        print(f'Zip ID found: {zip_id}')
        subprocess.run(['gdown', zip_id, '-O', ZIP_PATH], check=True)

print('✓ Zip present at', ZIP_PATH)

# ── Step 2: extract ────────────────────────────────────────────────────────
expected = os.path.join(DATASET_DIR, 'RoadAnomaly21')
if not os.path.isdir(expected):
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
    print('✓ Extracted')
else:
    print('✓ Already extracted')

# ── Step 3: verify tree ────────────────────────────────────────────────────
print('\nDataset tree:')
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    if level > 2: continue
    print('  ' * level + os.path.basename(root) + '/')
    if level == 2:
        exts = set(os.path.splitext(f)[1] for f in files)
        print('  ' * (level+1) + f'{len(files)} files  extensions: {exts}')

✓ Zip present at /content/Anomaly_Validation_Datasets.zip
Extracting...
✓ Extracted

Dataset tree:
datasets/
  __MACOSX/
    Validation_Dataset/
      1 files  extensions: {'.DS_Store'}
  Validation_Dataset/
    RoadObsticle21/
      1 files  extensions: {''}
    RoadAnomaly/
      1 files  extensions: {''}
    fs_static/
      1 files  extensions: {''}
    FS_LostFound_full/
      1 files  extensions: {''}
    RoadAnomaly21/
      1 files  extensions: {''}


## Cell 4 — Write `eval/evalAnomaly.py`
`%%writefile` saves this cell to disk — **never comment it out**.

In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py
import os, sys, glob, argparse
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from erfnet import ERFNet

NUM_CLASSES   = 20
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def load_model(path, device):
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Weights not found: {path}')
    model = ERFNet(NUM_CLASSES)
    state = torch.load(path, map_location=device)
    if 'state_dict' in state: state = state['state_dict']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    expected_missing = {'encoder.output_conv.weight', 'encoder.output_conv.bias'}
    real_missing = set(missing) - expected_missing
    if real_missing: raise RuntimeError(f'Unexpected missing keys: {real_missing}')
    return model.to(device).eval()

def preprocess(path):
    img = Image.open(path).convert('RGB')
    t = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
    return t(img).unsqueeze(0)

@torch.no_grad()
def get_logits(model, tensor, device):
    return model(tensor.to(device)).squeeze(0)   # (C, H, W)

def score_msp(logits):
    return (1.0 - F.softmax(logits, dim=0).max(dim=0).values).cpu().numpy()
def score_maxlogit(logits):
    return (-logits.max(dim=0).values).cpu().numpy()
def score_maxentropy(logits):
    p = F.softmax(logits, dim=0)
    return (-(p * torch.log(p.clamp(min=1e-10))).sum(dim=0)).cpu().numpy()

METHODS = {'msp': score_msp, 'maxlogit': score_maxlogit, 'maxentropy': score_maxentropy}

def find_images(pattern):
    paths = sorted(glob.glob(pattern))
    if not paths: paths = sorted(glob.glob(pattern.replace('*.png','*.jpg')))
    if not paths: paths = sorted(glob.glob(pattern.replace('*.png','*.*')))
    return paths

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--input',       required=True)
    ap.add_argument('--method',      default='msp', choices=METHODS)
    ap.add_argument('--weights',     default='../trained_models/erfnet_pretrained.pth')
    ap.add_argument('--output',      default='./scores')
    ap.add_argument('--save-logits', action='store_true',  # ← NEW FLAG
        help='Also save raw (C,H,W) logits as *_logits.npy')
    args = ap.parse_args()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = load_model(args.weights, device)
    paths  = find_images(args.input)
    print(f'[INFO] Device={device}  Method={args.method}  Images={len(paths)}')
    os.makedirs(args.output, exist_ok=True)
    fn = METHODS[args.method]

    for i, p in enumerate(paths):
        stem   = os.path.splitext(os.path.basename(p))[0]
        logits = get_logits(model, preprocess(p), device)
        if args.save_logits:                              # ← SAVE LOGITS
            np.save(os.path.join(args.output, f'{stem}_logits.npy'),
                    logits.cpu().numpy().astype(np.float32))
        score  = fn(logits)
        np.save(os.path.join(args.output, f'{stem}_{args.method}.npy'), score)
        if (i+1) % 10 == 0 or (i+1) == len(paths):
            print(f'  [{i+1}/{len(paths)}]')
    print(f'[DONE] Saved to {args.output}')

if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py


## Cell 5 — Write `eval/metrics.py`

In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/metrics.py
# metrics.py — AuPRC and FPR@95 for anomaly segmentation evaluation
import os, glob
import numpy as np
from PIL import Image
from sklearn.metrics import average_precision_score, roc_curve


def load_pairs(scores_dir, masks_dir, method, mask_ext=None):
    """
    Load matching score maps (.npy) and ground-truth masks.

    Mask convention (SMIYC / FS datasets):
      255  → void / ignore
      > 0  → anomaly  (positive)
      0    → normal   (negative)
    """
    npy_files = sorted(glob.glob(os.path.join(scores_dir, f'*_{method}.npy')))
    if not npy_files:
        raise FileNotFoundError(
            f'No *_{method}.npy files in {scores_dir}')

    # Auto-detect mask extension if not given
    if mask_ext is None:
        for ext in ('png', 'jpg', 'jpeg', 'webp'):
            sample = os.path.join(
                masks_dir,
                os.path.basename(npy_files[0]).replace(f'_{method}.npy', f'.{ext}'))
            if os.path.isfile(sample):
                mask_ext = ext
                break
        if mask_ext is None:
            mask_ext = 'png'

    S, L = [], []
    for npy_path in npy_files:
        stem      = os.path.basename(npy_path).replace(f'_{method}.npy', '')
        mask_path = os.path.join(masks_dir, f'{stem}.{mask_ext}')

        if not os.path.isfile(mask_path):
            print(f'  [SKIP] mask not found: {mask_path}')
            continue

        score = np.load(npy_path).astype(np.float32)
        mask  = np.array(Image.open(mask_path))

        # Resize score to mask resolution if they differ
        if score.shape != mask.shape:
            score = np.array(
                Image.fromarray(score).resize(
                    (mask.shape[1], mask.shape[0]), Image.BILINEAR))

        valid = mask != 255
        if valid.sum() == 0:
            print(f'  [SKIP] all void: {mask_path}')
            continue

        S.append(score[valid].ravel())
        L.append((mask[valid] > 0).ravel().astype(np.int32))

    if not S:
        raise RuntimeError(f'No valid (score, mask) pairs loaded from {scores_dir}')

    return np.concatenate(S), np.concatenate(L)


def compute_auprc(scores, labels):
    """Area under Precision-Recall Curve (0–1)."""
    return float(average_precision_score(labels, scores))


def fpr95(scores, labels):
    """False Positive Rate at 95 % True Positive Rate (0–1)."""
    fpr_arr, tpr_arr, _ = roc_curve(labels, scores)
    idx = np.searchsorted(tpr_arr, 0.95)
    return float(fpr_arr[min(idx, len(fpr_arr) - 1)])

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/metrics.py


## Cell 6 — Configuration
Paths match the **exact** folder names from the downloaded zip.

In [ ]:
import os

REPO_ROOT    = '/content/MaskArchitectureAnomaly_CourseProject'
WEIGHTS      = f'{REPO_ROOT}/trained_models/erfnet_pretrained.pth'
EVAL_SCRIPT  = f'{REPO_ROOT}/eval/evalAnomaly.py'
RESULTS_ROOT = '/content/drive/MyDrive/AnomalyProject/results'
DATASET_BASE = '/content/datasets/Validation_Dataset'

# ── Exact folder names confirmed from the zip ─────────────────────────────
# Each entry: dataset_key → (image_glob, masks_dir)
DATASETS = {
    'RoadAnomaly21': (
        f'{DATASET_BASE}/RoadAnomaly21/images/*.png',
        f'{DATASET_BASE}/RoadAnomaly21/labels_masks'),
    'RoadObsticle21': (
        f'{DATASET_BASE}/RoadObsticle21/images/*.webp',
        f'{DATASET_BASE}/RoadObsticle21/labels_masks'),
    'FS_LostFound': (
        f'{DATASET_BASE}/FS_LostFound_full/images/*.png',
        f'{DATASET_BASE}/FS_LostFound_full/labels_masks'),
    'fs_static': (
        f'{DATASET_BASE}/fs_static/images/*.jpg',
        f'{DATASET_BASE}/fs_static/labels_masks'),
    'RoadAnomaly': (
        f'{DATASET_BASE}/RoadAnomaly/images/*.jpg',
        f'{DATASET_BASE}/RoadAnomaly/labels_masks'),
}

METHODS = ['msp', 'maxlogit', 'maxentropy']

# ── Sanity check: verify all image folders exist ──────────────────────────
print('Dataset verification:')
import glob as _glob
for ds, (img_glob, mask_dir) in DATASETS.items():
    imgs  = _glob.glob(img_glob)
    # also try jpg if no png found
    if not imgs:
        imgs = _glob.glob(img_glob.replace('*.png','*.jpg', '*.webp'))
    masks = os.path.isdir(mask_dir)
    status = '✓' if imgs and masks else '✗'
    print(f'  {status} {ds:20s}  images={len(imgs):4d}  masks_dir={masks}')
    if imgs:
        # Update glob to correct extension for later use
        ext = os.path.splitext(imgs[0])[1]
        DATASETS[ds] = (img_glob.replace('*.png', f'*{ext}'), mask_dir)

Dataset verification:
  ✓ RoadAnomaly21         images=  10  masks_dir=True
  ✓ RoadObsticle21        images=  30  masks_dir=True
  ✓ FS_LostFound          images= 100  masks_dir=True
  ✓ fs_static             images=  30  masks_dir=True
  ✓ RoadAnomaly           images=  60  masks_dir=True


## Cell 7 — Run scoring (saves .npy maps to Drive)

In [ ]:
import subprocess, sys, os

os.makedirs(RESULTS_ROOT, exist_ok=True)

for method in METHODS:
    for ds_name, (img_glob, _) in DATASETS.items():
        out_dir = os.path.join(RESULTS_ROOT, f'{method}_{ds_name}')
        os.makedirs(out_dir, exist_ok=True)
        print(f'\n► {method:12s}  {ds_name}')

        r = subprocess.run(
            [sys.executable, EVAL_SCRIPT,
             '--input',   img_glob,
             '--method',  method,
             '--weights', WEIGHTS,
             '--output',  out_dir],
            capture_output=True, text=True
        )
        out = (r.stdout + r.stderr).strip()
        print(out[-500:] if len(out) > 500 else out)
        if r.returncode != 0:
            print(f'  ⚠ Exit code {r.returncode}')

print('\n✅ Scoring complete.')


► msp           RoadAnomaly21
[INFO] Device=cpu  Method=msp  Images=10
  [10/10]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_RoadAnomaly21

► msp           RoadObsticle21
[INFO] Device=cpu  Method=msp  Images=30
  [10/30]
  [20/30]
  [30/30]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_RoadObsticle21

► msp           FS_LostFound
[INFO] Device=cpu  Method=msp  Images=100
  [10/100]
  [20/100]
  [30/100]
  [40/100]
  [50/100]
  [60/100]
  [70/100]
  [80/100]
  [90/100]
  [100/100]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_FS_LostFound

► msp           fs_static
[INFO] Device=cpu  Method=msp  Images=30
  [10/30]
  [20/30]
  [30/30]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_fs_static

► msp           RoadAnomaly
[INFO] Device=cpu  Method=msp  Images=60
  [10/60]
  [20/60]
  [30/60]
  [40/60]
  [50/60]
  [60/60]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_RoadAnomaly

► maxlogi

## Cell 8 — Compute metrics & print results table

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()   # select temperature_scaling.py from your computer

# Move it to /content so Python can import it
for fname in uploaded:
    shutil.move(fname, f'/content/{fname}')
    print(f'✓ Uploaded: {fname}')

import sys
sys.path.insert(0, '/content')
print('✓ /content added to sys.path')

Saving temperature_scaling.py to temperature_scaling.py
✓ Uploaded: temperature_scaling.py
✓ /content added to sys.path


In [ ]:
import subprocess, sys, os

LOGITS_ROOT = '/content/drive/MyDrive/AnomalyProject/logits'
os.makedirs(LOGITS_ROOT, exist_ok=True)

for ds_name, (img_glob, _) in DATASETS.items():
    out_dir = os.path.join(LOGITS_ROOT, ds_name)
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n► Saving logits: {ds_name}')
    r = subprocess.run(
        [sys.executable, EVAL_SCRIPT,
         '--input',       img_glob,
         '--method',      'msp',       # method doesn't matter, we want logits
         '--weights',     WEIGHTS,
         '--output',      out_dir,
         '--save-logits'],             # ← the new flag
        capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out[-400:] if len(out) > 400 else out)
    if r.returncode != 0:
        print(f'  ⚠ Exit code {r.returncode}')

print('\n✅ Logits saved to', LOGITS_ROOT)


► Saving logits: RoadAnomaly21
[INFO] Device=cpu  Method=msp  Images=10
  [10/10]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/logits/RoadAnomaly21

► Saving logits: RoadObsticle21
[INFO] Device=cpu  Method=msp  Images=30
  [10/30]
  [20/30]
  [30/30]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/logits/RoadObsticle21

► Saving logits: FS_LostFound
[INFO] Device=cpu  Method=msp  Images=100
  [10/100]
  [20/100]
  [30/100]
  [40/100]
  [50/100]
  [60/100]
  [70/100]
  [80/100]
  [90/100]
  [100/100]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/logits/FS_LostFound

► Saving logits: fs_static
[INFO] Device=cpu  Method=msp  Images=30
  [10/30]
  [20/30]
  [30/30]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/logits/fs_static

► Saving logits: RoadAnomaly
[INFO] Device=cpu  Method=msp  Images=60
  [10/60]
  [20/60]
  [30/60]
  [40/60]
  [50/60]
  [60/60]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/logits/RoadAnomaly

✅ Logits saved to /content/dr

In [ ]:
import os
import sys
import gc  # Import garbage collection to prevent crashes
import torch # Or numpy, whichever your backend uses for clearing cache

sys.path.insert(0, '/content')
from temperature_scaling import evaluate_temperature_scaling

LOGITS_ROOT = '/content/drive/MyDrive/AnomalyProject/logits'

FIXED_TEMPS  = [1.0, 0.5, 0.75, 1.1]   # 1.0 = standard MSP baseline

# OPTION A: Shortened search (Just 5 values instead of 30 to prevent crashes)
SEARCH_TEMPS = [0.6, 0.7, 0.8, 0.9, 1.2]


# ── Map dataset names to (logits_dir, masks_dir) ──────────────────────────
EVAL_DATASETS = {
    'SMIYC RA-21':    ('RoadAnomaly21',  'RoadAnomaly21'),
    'SMIYC RO-21':    ('RoadObsticle21', 'RoadObsticle21'),
    'FS L&F':         ('FS_LostFound',   'FS_LostFound_full'),
    'FS Static':      ('fs_static',      'fs_static'),
    'Road Anomaly':   ('RoadAnomaly',    'RoadAnomaly'),
}
DATASET_BASE = '/content/datasets/Validation_Dataset'

# ── Run evaluation ────────────────────────────────────────────────────────
all_results = {}
for display_name, (logits_sub, mask_sub) in EVAL_DATASETS.items():
    logits_dir = os.path.join(LOGITS_ROOT, logits_sub)
    masks_dir  = os.path.join(DATASET_BASE, mask_sub, 'labels_masks')
    print(f'\n{"="*60}')
    print(f'  Dataset: {display_name}')
    print(f'{"="*60}')
    try:
        all_results[display_name] = evaluate_temperature_scaling(
            logits_dir   = logits_dir,
            masks_dir    = masks_dir,
            fixed_temps  = FIXED_TEMPS,
            search_temps = SEARCH_TEMPS,
            metric       = 'auprc',
        )
    except Exception as e:
        print(f'  ERROR: {e}')
        all_results[display_name] = None

    # CRITICAL: Clear memory after each dataset processing loop
    gc.collect()
    if 'torch' in sys.modules:
        torch.cuda.empty_cache()

# ── Print the table ───────────────────────────────────────────────────────
DS_NAMES = list(EVAL_DATASETS.keys())
T_LABELS = {1.0: 'MSP', 0.5: 'MSP(t=0.5)', 0.75: 'MSP(t=0.75)', 1.1: 'MSP(t=1.1)'}

print('\n\n' + '='*120)
header = f"{'Method':<16}" + ''.join(f"  {d:<22}" for d in DS_NAMES)
print(header)
subheader = f"{'':16}" + ''.join(f"  {'AuPRC':>9}  {'FPR95':>9}   " for _ in DS_NAMES)
print(subheader)
print('-'*120)

def get_row(t_val, label, results):
    row = f'{label:<16}'
    for ds in DS_NAMES:
        r = results.get(ds)
        if r is None:
            row += f"  {'—':>9}  {'—':>9}   "
            continue
        try:
            idx    = r['fixed']['t'].index(t_val)
            auprc  = r['fixed']['auprc'][idx] * 100
            fpr95  = r['fixed']['fpr95'][idx] * 100
            row   += f"  {auprc:>8.2f}%  {fpr95:>8.2f}%   "
        except ValueError:
            row += f"  {'?':>9}  {'?':>9}   "
    return row

for t_val, label in T_LABELS.items():
    print(get_row(t_val, label, all_results))

# Best T row calculation
best_row = f"{'MSP (best t)':<16}"
for ds in DS_NAMES:
    r = all_results.get(ds)
    if r is None:
        best_row += f"  {'—':>9}  {'—':>9}   "
    else:
        # Fallback logic: If search yielded data, use it; otherwise find max in fixed
        if 'search' in r and r['search'] is not None and 'best_auprc' in r['search']:
            auprc = r['search']['best_auprc'] * 100
            fpr95 = r['search']['best_fpr95'] * 100
        else:
            # Fallback to finding the highest AuPRC inside FIXED_TEMPS
            max_idx = r['fixed']['auprc'].index(max(r['fixed']['auprc']))
            auprc = r['fixed']['auprc'][max_idx] * 100
            fpr95 = r['fixed']['fpr95'][max_idx] * 100

        best_row += f"  {auprc:>8.2f}%  {fpr95:>8.2f}%   "
print(best_row)
print('-'*120)

# Print best T values
print('\nBest T per dataset:')
for ds in DS_NAMES:
    r = all_results.get(ds)
    if r:
        if 'search' in r and r['search'] is not None and 'best_t' in r['search']:
            print(f"  {ds:<20} T={r['search']['best_t']:.3f}")
        else:
            max_idx = r['fixed']['auprc'].index(max(r['fixed']['auprc']))
            print(f"  {ds:<20} T={r['fixed']['t'][max_idx]:.3f} (From Fixed Pool)")


  Dataset: SMIYC RA-21
Loaded 10 logit maps, 10 masks.

── Fixed temperatures ─────────────────────────────────────
  T=1.0000  AuPRC=15.80%  FPR95=89.40%
  T=0.5000  AuPRC=16.21%  FPR95=89.54%
  T=0.7500  AuPRC=16.02%  FPR95=89.23%
  T=1.1000  AuPRC=15.70%  FPR95=89.47%

── Temperature grid search ────────────────────────────────
  T=0.6000  AuPRC=16.15%  FPR95=89.21%
  T=0.7000  AuPRC=16.07%  FPR95=89.21%
  T=0.8000  AuPRC=15.98%  FPR95=89.26%
  T=0.9000  AuPRC=15.89%  FPR95=89.33%
  T=1.2000  AuPRC=15.60%  FPR95=89.55%

Best T (grid search): 0.600  →  AuPRC=16.15%  FPR95=89.21%

  Dataset: SMIYC RO-21
Loaded 30 logit maps, 30 masks.

── Fixed temperatures ─────────────────────────────────────
  T=1.0000  AuPRC=0.64%  FPR95=96.41%
  T=0.5000  AuPRC=0.62%  FPR95=100.00%
  T=0.7500  AuPRC=0.63%  FPR95=96.25%
  T=1.1000  AuPRC=0.64%  FPR95=96.48%

── Temperature grid search ────────────────────────────────
  T=0.6000  AuPRC=0.62%  FPR95=100.00%
  T=0.7000  AuPRC=0.63%  FPR95=96.33%
  T